# MedNet/Darma: model and code before the test

Classification maps a crop to two numbers. The actual class is Darma; its forward returns logits AND five stage activations. This demonstration initializes random weights, so scores have no assigned clinical meaning. Read docs/MODELS_AND_CODE_LESSONS.md first. Select a GPU runtime. GPU execution remains pending until these cells run successfully.

## 1. Obtain the exact model source and check GPU

In [ ]:
import sys, subprocess, json
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
DEVICE="cuda:0"
print("GPU:",torch.cuda.get_device_name(0),"PyTorch:",torch.__version__)
REPO=Path("/content/MedSAM_student")
if not REPO.exists():
    subprocess.run(["git","clone","--branch","codex/student-foundation","--single-branch",
                    "https://github.com/KrishnachaithanyaThummala/MedSAM.git",str(REPO)],check=True)
PROJECT=REPO/"student_project"
sys.path.insert(0,str(PROJECT))
MED=Path("/content/MedNet_student")
if not MED.exists(): subprocess.run(["git","clone","https://github.com/Md-Ferdous/MedNet.git",str(MED)],check=True)
subprocess.run(["git","checkout","--detach","6adb072630811b1d39568a87732aed96dc39c9ae"],cwd=MED,check=True)
print("Project commit:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())


**Result/inference:** Inspect the printed GPU, PyTorch and commit. They are evidence for this session.

## 2. Load the authors' architecture, excluding training-only imports

In [ ]:
from src.classification import build_mednet,classify
model,provenance=build_mednet(MED/"MedNet.py",DEVICE)
print(json.dumps(provenance,indent=2))
print("Parameters:",sum(p.numel() for p in model.parameters()))


**Result/inference:** Seven original architecture class bodies are compiled unchanged. No trained MedNet checkpoint is loaded. The import adaptation is documented.

## 3. Forward pass and feature shapes

In [ ]:
torch.set_num_threads(4)
x=torch.zeros(1,3,224,224)
report=classify(model,x)
print(json.dumps(report,indent=2))
assert report["output_shape"]==[1,2]
OUT=PROJECT/"outputs/mednet_gpu";OUT.mkdir(parents=True,exist_ok=True)
(OUT/"forward.json").write_text(json.dumps(report|provenance,indent=2))
(OUT/"requirements.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))


**Result/inference:** Expected five stage sizes: 224,112,56,28,14. A finite 1x2 output completes the forward-pass check; it does not validate a classifier.

## 4. Save the evidence

In [ ]:
import shutil
from google.colab import files
files.download(shutil.make_archive("/content/MedNet_forward_results","zip",OUT))


**Result/inference:** Keep the results and save this notebook with actual outputs. A prepared notebook without executed outputs does not meet the Colab pass check.